# 07 - Preliminary interpretability, U.S. training only

This notebook describes model behavior on validation folds from the official U.S. training partition. It does not use reserved test labels and does not choose a final production model. Because the tuning comparison is still a training-CV selection exercise, two clearly labeled conservative candidates are interpreted: Logistic Regression and HistGradientBoosting.

Permutation importance is computed on each held-out training fold and averaged. The displayed importance is for the raw input column being permuted, not a causal effect and not a claim about an individual campaign.


In [1]:
from pathlib import Path
import sys, warnings
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'preprocessing.py').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from utils.us_modeling import CONSERVATIVE_POLICY, development_cv, load_us_training_data, conservative_pipeline
SEED = 42
X_train, y_train, project_ids = load_us_training_data(ROOT)
assert len(X_train) == 26945 and project_ids.is_unique
assert set(y_train.unique()) == {0, 1}
assert CONSERVATIVE_POLICY == dict(include_timing=False, include_flags=False, include_reward=False, exclude_renamed=True)
assert not {'pledged', 'funded percentage', 'backers', 'comments', 'updates', 'status', 'target'}.intersection(X_train.columns)
ACTIVE_INPUTS = ['goal', 'duration', 'category', 'subcategory', 'location']
X_model = X_train[ACTIVE_INPUTS].copy()
print(f'validation population: {len(X_train):,} official U.S. training rows; no reserved labels loaded')


validation population: 26,945 official U.S. training rows; no reserved labels loaded


In [2]:
candidates = {
    'Logistic Regression': conservative_pipeline(LogisticRegression(max_iter=2000, solver='lbfgs'), 'onehot'),
    'HistGradientBoosting': conservative_pipeline(HistGradientBoostingClassifier(max_iter=200, learning_rate=0.06, random_state=SEED), 'target'),
}
fold_rows = []
importance_rows = []
for model_name, estimator in candidates.items():
    oof = np.full(len(y_train), np.nan)
    for fold, (fit_idx, valid_idx) in enumerate(development_cv().split(X_model, y_train), start=1):
        fitted = clone(estimator).fit(X_model.iloc[fit_idx], y_train.iloc[fit_idx])
        valid_x, valid_y = X_model.iloc[valid_idx], y_train.iloc[valid_idx]
        proba = fitted.predict_proba(valid_x)[:, 1]
        oof[valid_idx] = proba
        perm = permutation_importance(fitted, valid_x, valid_y, scoring='roc_auc', n_repeats=3, random_state=SEED + fold, n_jobs=1)
        for column, mean, sd in zip(valid_x.columns, perm.importances_mean, perm.importances_std):
            importance_rows.append({'model': model_name, 'fold': fold, 'feature': column, 'importance': mean, 'importance_sd': sd})
    pred = oof >= 0.5
    fold_rows.append({'model': model_name, 'oof_roc_auc': roc_auc_score(y_train, oof),
                      'oof_accuracy': accuracy_score(y_train, pred), 'oof_f1': f1_score(y_train, pred)})
metrics = pd.DataFrame(fold_rows).set_index('model')
importance = pd.DataFrame(importance_rows)
summary = (importance.groupby(['model', 'feature'], as_index=False)['importance'].mean()
           .sort_values(['model', 'importance'], ascending=[True, False]))
display(metrics.round(4))
display(summary.groupby('model').head(8).round(4))


,oof_roc_auc,oof_accuracy,oof_f1
model,,,
Logistic Regression,0.7198,0.6684,0.7111
HistGradientBoosting,0.7233,0.6684,0.7156


,model,feature,importance
2,HistGradientBoosting,goal,0.0902
4,HistGradientBoosting,subcategory,0.0619
3,HistGradientBoosting,location,0.0254
1,HistGradientBoosting,duration,0.0251
0,HistGradientBoosting,category,0.0020
7,Logistic Regression,goal,0.0870
8,Logistic Regression,location,0.0296
5,Logistic Regression,category,0.0283
9,Logistic Regression,subcategory,0.0249
6,Logistic Regression,duration,0.0149


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, model_name in zip(axes, candidates):
    top = summary[summary['model'].eq(model_name)].head(8).sort_values('importance')
    ax.barh(top['feature'], top['importance'], color='#2c6f52')
    ax.set_title(model_name + ' held-out-fold permutation importance')
    ax.set_xlabel('Mean ROC-AUC decrease after permutation')
plt.tight_layout()
print('These importances describe transformed pipeline behavior on training-validation folds.')
print('They do not show that changing a feature causes success, and location effects are not causal country/state comparisons.')


These importances describe transformed pipeline behavior on training-validation folds.
They do not show that changing a feature causes success, and location effects are not causal country/state comparisons.


The OOF metrics are useful for understanding the candidate models, but they are still development evidence. Feature importance is aggregated at the raw input-column level, so it does not separate the several transformed features derived from goal or categories. A later interpretability pass can add coefficient details or fold-specific partial dependence after the team agrees on the model-selection protocol.
